In [1]:
# Fill missing data in csv files with 0
# # some of the csv are missing data for some datetimes
# # i want to fill the missing data with 0 all data for the same datetime
# import pandas as pd
# import datetime
# from glob import glob
# import numpy

# # got a filename that has all the datetimes with prices
# filename = 'data/SP500.csv'
# #get the list of datetimes in the field timestamp with pandas

# df = pd.read_csv(filename)
# datetimes = df['timestamp'].to_list()

# # get the list of csv files
# files = glob('Data/*.csv')

# # This is a sample of the format of the csv files
# #timestamp,symbol,open,high,low,close,volume,trade_count,vwap
# #2016-06-01 09:30:00-04:00,QSII,12.69,12.74,12.65,12.67,48037.0,208.0,12.706119

# # for each file in the list: 
# # create a new file with the same name +"_filled" , put all the datetimes in the first column with the same format
# # timestamp,symbol,open,high,low,close,volume,trade_count,vwap
# # keep the same order of the columns 
# # if exists data for a datetime, keep the data
# # if there is no data for a datetime, fill the data with 0
# # process the files with pandas

# for file in files:
#     print('Processing file', file)
#     df = pd.read_csv(file)
#     #df['timestamp'] = pd.to_datetime(df['timestamp'])
#     df = df.set_index('timestamp')
#     df.drop_duplicates(keep='first', inplace=True)
#     df = df.reindex(datetimes, fill_value=0)
#     df = df.reset_index()
#     df.to_csv(file.replace('Data/', 'Data/filled_'), index=False)
#     print('File', file, 'processed')

In [2]:
# #Convert to parquet
# from glob import glob
# import pandas as pd


# files = glob('Data/*.csv')

# for file in files:
#     parquet_file_name = file.replace('.csv', '.pk')
#     print(f'Processing file: {file} to {parquet_file_name}')
#     df = pd.read_csv(file, index_col='timestamp', parse_dates=True)
#     # Drop column symbol
#     df.drop(columns='symbol', inplace=True)
#     if parquet_file_name[-3:] == '.pk':
#         df.to_parquet(parquet_file_name, index=True)
#         print("The file was saved as parquet: {parquet_file_name}")

In [3]:
import pandas as pd
from datetime import datetime

# Danh sách các cổ phiếu trong Dow Jones 30
tickers = ['AAPL','ABCB','ABNB','BFR','BRCD','CSCO','DAL','EAST','ESPR','F',
           'FFIE','GOOGL','JPM','JTAI','META','MSFT','NVDA','PFE','PG','PSX',
           'QSII', 'SGLY','SP500','TSLA','UNH','USA','WMT','XOM']

# ticker of benchmark
benchmark = 'SP500'

# Get the data from the CSV files
stock_data = {}
for ticker in tickers:
    df = pd.read_parquet(f'Data/{ticker}.pk')
    #pass the columns to capitalize
    df.columns = df.columns.str.capitalize()
    stock_data[ticker] = df


# Establish the time range for the data
#trainInit = datetime.fromisoformat('2016-06-01 00:00:01-04:00')
trainInit = datetime.fromisoformat('2019-06-01 00:00:01-04:00')
trainEnd = datetime.fromisoformat('2020-06-01 00:00:01-04:00')
testInit = datetime.fromisoformat('2020-06-02 00:00:01-04:00')
testEnd = datetime.fromisoformat('2021-06-01 00:00:01-04:00')
#testEnd = datetime.fromisoformat('2024-03-01 00:00:01-04:00')

# split the data into training, validation and test sets
training_data_time_range = (trainInit, trainEnd)
test_data_time_range = (testInit, testEnd)

# split the data into training, validation and test sets
training_data = {}
validation_data = {}
test_data = {}

for ticker, df in stock_data.items():
    training_data[ticker] = df.loc[training_data_time_range[0]:training_data_time_range[1]]
    test_data[ticker] = df.loc[test_data_time_range[0]:test_data_time_range[1]]

# print shape of training, validation and test data
ticker = 'AAPL'
print(f'Training data shape for {ticker}: {training_data[ticker].shape}')
print(f'Test data shape for {ticker}: {test_data[ticker].shape}')

# Display the first 5 rows of the data
stock_data['AAPL'].head()

Training data shape for AAPL: (15595, 7)
Test data shape for AAPL: (15664, 7)


,Open,High,Low,Close,Volume,Trade_count,Vwap
timestamp,,,,,,,
2016-06-01 04:00:00-04:00,99.88,99.98,99.88,99.98,517.0,9.0,99.957447
2016-06-01 04:15:00-04:00,0.00,0.00,0.00,0.00,0.0,0.0,0.000000
2016-06-01 04:30:00-04:00,0.00,0.00,0.00,0.00,0.0,0.0,0.000000
2016-06-01 04:45:00-04:00,99.61,99.61,99.30,99.50,1490.0,10.0,99.497651
2016-06-01 05:00:00-04:00,99.38,99.52,99.38,99.52,527.0,4.0,99.425863


In [4]:
import pandas as pd
import numpy as np

def add_technical_indicators(df):
    # calculate RSI 14 
    delta = df['Close'].diff()
    up = delta.where(delta > 0, 0)
    down = -delta.where(delta < 0, 0)
    rs = up.rolling(window=14).mean() / down.rolling(window=14).mean()
    df['RSI'] = 100 - (100 / (1 + rs))

    # EMA 12 / 26  en MACD
    df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = df['EMA12'] - df['EMA26']
    df['Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # RSI 14
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # CCI 20
    tp = (df['High'] + df['Low'] + df['Close']) / 3
    sma_tp = tp.rolling(window=20).mean()
    mean_dev = tp.rolling(window=20).apply(lambda x: np.mean(np.abs(x - x.mean())))
    df['CCI'] = (tp - sma_tp) / (0.015 * mean_dev)
    
    # ADX 14
    high_diff = df['High'].diff()
    low_diff = df['Low'].diff()
    df['+DM'] = np.where((high_diff > low_diff) & (high_diff > 0), high_diff, 0)
    df['-DM'] = np.where((low_diff > high_diff) & (low_diff > 0), low_diff, 0)
    tr = pd.concat([df['High'] - df['Low'], np.abs(df['High'] - df['Close'].shift(1)), np.abs(df['Low'] - df['Close'].shift(1))], axis=1).max(axis=1)
    atr = tr.ewm(span=14, adjust=False).mean()
    df['+DI'] = 100 * (df['+DM'].ewm(span=14, adjust=False).mean() / atr)
    df['-DI'] = 100 * (df['-DM'].ewm(span=14, adjust=False).mean() / atr)
    dx = 100 * np.abs(df['+DI'] - df['-DI']) / (df['+DI'] + df['-DI'])
    df['ADX'] = dx.ewm(span=14, adjust=False).mean()

    # drop NaN values
    df.dropna(inplace=True)

    # keep Open, High, Low, Close, Volume, MACD, Signal, RSI, CCI, ADX
    df = df[['Open', 'High', 'Low', 'Close', 'Volume', 'MACD', 'Signal', 'RSI', 'CCI', 'ADX']]

    return df

In [5]:
# add technical indicators to the training data for each stock
for ticker, df in training_data.items():
    training_data[ticker] = add_technical_indicators(df)

# add technical indicators to the test data for each stock
for ticker, df in test_data.items():
    test_data[ticker] = add_technical_indicators(df)


# Create benchmark data
benchmark_training_data = training_data[benchmark]
benchmark_test_data = test_data[benchmark]

# print the first 5 rows of the data
print('Shape of training data for AAPL:', training_data['AAPL'].shape)
print('Shape of test data for AAPL:', test_data['AAPL'].shape)

# print the first 5 rows of the benchmark data
print('Shape of training benchmark data:', benchmark_training_data.shape)
print('Shape of test benchmark data:', benchmark_test_data.shape)

C:\Users\tcorn\AppData\Local\Temp\ipykernel_24660\3622855247.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['RSI'] = 100 - (100 / (1 + rs))
C:\Users\tcorn\AppData\Local\Temp\ipykernel_24660\3622855247.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
C:\Users\tcorn\AppData\Local\Temp\ipykernel_24660\3622855247.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,co

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\tcorn\Desktop\RLTesis\DRLST\lib\site-packages\IPython\core\interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\tcorn\AppData\Local\Temp\ipykernel_24660\3700184032.py", line 3, in <module>
    training_data[ticker] = add_technical_indicators(df)
  File "C:\Users\tcorn\AppData\Local\Temp\ipykernel_24660\3622855247.py", line 28, in add_technical_indicators
    mean_dev = tp.rolling(window=20).apply(lambda x: np.mean(np.abs(x - x.mean())))
  File "c:\Users\tcorn\Desktop\RLTesis\DRLST\lib\site-packages\pandas\core\window\rolling.py", line 1913, in apply
    return super().apply(
  File "c:\Users\tcorn\Desktop\RLTesis\DRLST\lib\site-packages\pandas\core\window\rolling.py", line 1390, in apply
    return self._apply(
  File "c:\Users\tcorn\Desktop\RLTesis\DRLST\lib\site-packages\pandas\core\window\rolling.py", line 615, in _apply
    return self._apply_blockwise(homogeneous_func